# 1️⃣ Introducción

En este cuaderno exploramos un enfoque sencillo para predecir si un equipo ganará (W) o perderá (L) un partido de la NBA.

- Trabajamos exclusivamente con estadísticas históricas calculadas como medias móviles de los últimos 10 partidos (`ROLL10_...`).
- No diferenciamos entre partidos como local o visitante; cada registro representa el rendimiento general del equipo antes de cada encuentro.
- Permitimos seleccionar una fecha de corte para entrenar con partidos anteriores y evaluar el modelo con encuentros posteriores, de forma que podamos probar distintos momentos de la temporada.


# 2️⃣ Importación de librerías

Importamos las dependencias principales para manipular datos, entrenar modelos y visualizar resultados.


In [ ]:
import datetime as dt
from typing import Dict, List

import numpy as np
import pandas as pd

from sklearn.calibration import calibration_curve
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    log_loss,
    roc_auc_score,
    roc_curve,
)
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns

import ipywidgets as widgets

sns.set_style("whitegrid")


# 3️⃣ Carga de datos

Leemos el archivo `Parquet` con los *team gamelogs*, verificamos que exista la columna objetivo `WL_NUM`, convertimos la fecha del partido y conservamos únicamente las columnas necesarias (`WL_NUM` y los indicadores `ROLL10_...`).


In [ ]:
DATA_PATH = "/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/teamgamelogs_by_game.parquet"

df_raw = pd.read_parquet(DATA_PATH)
df_raw["GAME_DATE"] = pd.to_datetime(df_raw["GAME_DATE"], errors="coerce")

if "WL_NUM" not in df_raw.columns:
    raise KeyError("El dataset debe contener la columna 'WL_NUM'.")

feature_cols: List[str] = sorted([col for col in df_raw.columns if col.startswith("ROLL10_")])
columns_to_keep = ["GAME_DATE", "TEAM_ID", "TEAM_ABBREVIATION", "WL_NUM"] + feature_cols

df_model = df_raw.loc[:, columns_to_keep].copy()
df_model = df_model.dropna(subset=["GAME_DATE", "WL_NUM"])
df_model = df_model.sort_values(["TEAM_ID", "GAME_DATE"]).reset_index(drop=True)

df_model["WL_NUM"] = df_model["WL_NUM"].astype(float)
df_model = df_model.dropna(subset=feature_cols + ["WL_NUM"])

n_rows, n_cols = df_model.shape
feature_summary = pd.Series(feature_cols, name="Features ROLL10")

date_min = df_model["GAME_DATE"].min()
date_max = df_model["GAME_DATE"].max()

print(f"Registros disponibles: {n_rows:,} filas x {n_cols} columnas")
print(f"Número de features ROLL10: {len(feature_cols)}")
print(f"Rango temporal: {date_min.date()} → {date_max.date()}")
feature_summary.head(10)


# 4️⃣ Selección de fecha de evaluación

Usamos un widget para fijar una fecha de corte (`cutoff_date`). Los partidos anteriores a esa fecha alimentan el entrenamiento y los posteriores sirven para evaluar el modelo. También mostramos la cantidad de partidos y la distribución de victorias/derrotas en cada subconjunto.


In [ ]:
cutoff_state: Dict[str, object] = {}

def describe_split(df_train: pd.DataFrame, df_test: pd.DataFrame) -> pd.DataFrame:
    def summary_block(df_part: pd.DataFrame) -> Dict[str, object]:
        total = len(df_part)
        wins = df_part["WL_NUM"].sum()
        losses = total - wins
        win_rate = wins / total if total else np.nan
        return {
            "partidos": int(total),
            "victorias": int(wins),
            "derrotas": int(losses),
            "win_rate": round(win_rate, 3) if pd.notnull(win_rate) else np.nan,
        }

    return pd.DataFrame({
        "Entrenamiento": summary_block(df_train),
        "Evaluación": summary_block(df_test),
    })

cutoff_input = widgets.Text(
    value="2025-02-15",
    description="Fecha corte",
    placeholder="YYYY-MM-DD",
    layout=widgets.Layout(width="250px"),
)

split_output = widgets.Output()


def apply_cutoff(change=None):
    with split_output:
        split_output.clear_output()
        try:
            cutoff_date = pd.to_datetime(cutoff_input.value).tz_localize(None)
        except (ValueError, TypeError):
            print("Introduce una fecha válida en formato YYYY-MM-DD.")
            return

        df_train = df_model[df_model["GAME_DATE"] < cutoff_date].copy()
        df_test = df_model[df_model["GAME_DATE"] >= cutoff_date].copy()

        if df_train.empty or df_test.empty:
            print("La selección de fecha debe dejar datos tanto para entrenamiento como para evaluación.")
            return

        X_train = df_train[feature_cols].copy()
        y_train = df_train["WL_NUM"].astype(int).copy()
        X_test = df_test[feature_cols].copy()
        y_test = df_test["WL_NUM"].astype(int).copy()

        cutoff_state.clear()
        cutoff_state.update({
            "cutoff_date": cutoff_date,
            "df_train": df_train,
            "df_test": df_test,
            "X_train": X_train,
            "y_train": y_train,
            "X_test": X_test,
            "y_test": y_test,
        })

        summary_df = describe_split(df_train, df_test)
        print(f"Fecha de corte seleccionada: {cutoff_date.date()}")
        display(summary_df)


cutoff_input.observe(apply_cutoff, names="value")

apply_cutoff()
display(widgets.VBox([cutoff_input, split_output]))


# 5️⃣ Entrenamiento del modelo

Entrenamos dos modelos básicos sobre las características `ROLL10_...`:

- **Regresión logística** con estandarización previa de las variables.
- **Gradient Boosting** con parámetros iniciales sencillos.

Evaluamos cada modelo mediante *accuracy*, *balanced accuracy*, *Brier score*, *log loss* y *ROC AUC*.


In [ ]:
if not cutoff_state:
    raise RuntimeError("Primero selecciona una fecha de corte válida para generar los conjuntos de entrenamiento y evaluación.")

X_train = cutoff_state["X_train"]
y_train = cutoff_state["y_train"]
X_test = cutoff_state["X_test"]
y_test = cutoff_state["y_test"]

metrics_records = []
trained_models: Dict[str, object] = {}
model_outputs: Dict[str, Dict[str, np.ndarray]] = {}

pipeline_lr = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "clf",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)
pipeline_lr.fit(X_train, y_train)
y_pred_lr = pipeline_lr.predict(X_test)
y_proba_lr = pipeline_lr.predict_proba(X_test)[:, 1]

metrics_records.append({
    "Modelo": "Regresión Logística",
    "Accuracy": accuracy_score(y_test, y_pred_lr),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred_lr),
    "Brier": brier_score_loss(y_test, y_proba_lr),
    "Log Loss": log_loss(y_test, y_proba_lr),
    "ROC AUC": roc_auc_score(y_test, y_proba_lr),
})
trained_models["Regresión Logística"] = pipeline_lr
model_outputs["Regresión Logística"] = {"y_pred": y_pred_lr, "y_proba": y_proba_lr}

model_gb = GradientBoostingClassifier(
    random_state=42,
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
)
model_gb.fit(X_train, y_train)
y_pred_gb = model_gb.predict(X_test)
y_proba_gb = model_gb.predict_proba(X_test)[:, 1]

metrics_records.append({
    "Modelo": "Gradient Boosting",
    "Accuracy": accuracy_score(y_test, y_pred_gb),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred_gb),
    "Brier": brier_score_loss(y_test, y_proba_gb),
    "Log Loss": log_loss(y_test, y_proba_gb),
    "ROC AUC": roc_auc_score(y_test, y_proba_gb),
})
trained_models["Gradient Boosting"] = model_gb
model_outputs["Gradient Boosting"] = {"y_pred": y_pred_gb, "y_proba": y_proba_gb}

metrics_df = pd.DataFrame(metrics_records).set_index("Modelo")

cutoff_state["trained_models"] = trained_models
cutoff_state["model_outputs"] = model_outputs
cutoff_state["metrics_df"] = metrics_df

metrics_df


# 6️⃣ Resultados y visualización

Comparamos las métricas de ambos modelos, dibujamos las curvas ROC y de calibración, y mostramos las matrices de confusión para analizar errores típicos.


In [ ]:
if "metrics_df" not in cutoff_state:
    raise RuntimeError("Entrena los modelos antes de visualizar resultados.")

metrics_df = cutoff_state["metrics_df"]
model_outputs = cutoff_state["model_outputs"]

display(metrics_df.style.format({
    "Accuracy": "{:.3f}",
    "Balanced Accuracy": "{:.3f}",
    "Brier": "{:.3f}",
    "Log Loss": "{:.3f}",
    "ROC AUC": "{:.3f}",
}))

plt.figure(figsize=(8, 6))
for model_name, outputs in model_outputs.items():
    fpr, tpr, _ = roc_curve(cutoff_state["y_test"], outputs["y_proba"])
    plt.plot(fpr, tpr, label=f"{model_name} (AUC={metrics_df.loc[model_name, 'ROC AUC']:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.6)
plt.title("Curvas ROC")
plt.xlabel("Tasa de falsos positivos")
plt.ylabel("Tasa de verdaderos positivos")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
for model_name, outputs in model_outputs.items():
    prob_pred, prob_true = calibration_curve(cutoff_state["y_test"], outputs["y_proba"], n_bins=10)
    plt.plot(prob_pred, prob_true, marker="o", label=model_name)
plt.plot([0, 1], [0, 1], "k--", alpha=0.6)
plt.title("Curvas de calibración")
plt.xlabel("Probabilidad predicha")
plt.ylabel("Proporción real de victorias")
plt.legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (model_name, outputs) in zip(axes, model_outputs.items()):
    cm = confusion_matrix(cutoff_state["y_test"], outputs["y_pred"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["L", "W"])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f"Matriz de confusión - {model_name}")
plt.tight_layout()
plt.show()


# 7️⃣ Selección de equipos y predicción por fecha

Creamos un panel interactivo para elegir una fecha posterior a la de corte y estimar la probabilidad de victoria de cada equipo con datos disponibles ese día, empleando el modelo seleccionado.


In [ ]:
if "trained_models" not in cutoff_state:
    raise RuntimeError("Entrena los modelos antes de generar predicciones interactivas.")

model_selector = widgets.ToggleButtons(
    options=list(cutoff_state["trained_models"].keys()),
    description="Modelo",
    layout=widgets.Layout(width="350px"),
)

prediction_date_input = widgets.Text(
    value=(cutoff_state["cutoff_date"] + pd.Timedelta(days=7)).strftime("%Y-%m-%d"),
    description="Fecha",
    placeholder="YYYY-MM-DD",
    layout=widgets.Layout(width="250px"),
)

prediction_output = widgets.Output()


def update_predictions(change=None):
    with prediction_output:
        prediction_output.clear_output()
        try:
            target_date = pd.to_datetime(prediction_date_input.value).tz_localize(None)
        except (ValueError, TypeError):
            print("Introduce una fecha válida en formato YYYY-MM-DD.")
            return

        if target_date < cutoff_state["cutoff_date"]:
            print("La fecha de predicción debe ser posterior a la fecha de corte utilizada para el split.")
            return

        df_eval = cutoff_state["df_test"]
        df_day = df_eval[df_eval["GAME_DATE"] == target_date].copy()

        if df_day.empty:
            print("No se encontraron partidos registrados para la fecha indicada en el conjunto de evaluación.")
            available_dates = df_eval["GAME_DATE"].drop_duplicates().sort_values()
            print("Prueba con una de estas fechas disponibles:")
            display(available_dates.tail(10).dt.strftime("%Y-%m-%d").to_frame(name="GAME_DATE"))
            return

        model_name = model_selector.value
        estimator = cutoff_state["trained_models"][model_name]

        proba = estimator.predict_proba(df_day[feature_cols])[:, 1]
        result_df = df_day[["TEAM_ABBREVIATION", "GAME_DATE"]].copy()
        result_df["Probabilidad_W"] = proba
        result_df = result_df.sort_values("Probabilidad_W", ascending=False).reset_index(drop=True)
        result_df["GAME_DATE"] = result_df["GAME_DATE"].dt.strftime("%Y-%m-%d")

        display(result_df)


model_selector.observe(update_predictions, names="value")
prediction_date_input.observe(update_predictions, names="value")

update_predictions()
display(widgets.VBox([widgets.HBox([model_selector, prediction_date_input]), prediction_output]))


# 8️⃣ Importancias de variables

Analizamos las 15 características más influyentes para cada modelo utilizando los coeficientes absolutos (regresión logística) y la importancia de las características (gradient boosting).


In [ ]:
trained_models = cutoff_state.get("trained_models")
if trained_models is None:
    raise RuntimeError("Entrena los modelos antes de revisar las importancias.")

coef_lr = trained_models["Regresión Logística"].named_steps["clf"].coef_[0]
importance_lr = (
    pd.DataFrame({"feature": feature_cols, "importance": np.abs(coef_lr)})
    .sort_values("importance", ascending=False)
    .head(15)
)

importance_gb = (
    pd.DataFrame({
        "feature": feature_cols,
        "importance": trained_models["Gradient Boosting"].feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .head(15)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].barh(importance_lr["feature"], importance_lr["importance"], color="#1f77b4")
axes[0].set_title("Top 15 coeficientes absolutos - Regresión Logística")
axes[0].invert_yaxis()

axes[1].barh(importance_gb["feature"], importance_gb["importance"], color="#ff7f0e")
axes[1].set_title("Top 15 importancias - Gradient Boosting")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()


# 9️⃣ Conclusiones y próximos pasos

En este cuaderno construimos una primera versión de un modelo de victorias/derrotas basado únicamente en estadísticas `ROLL10_`. El análisis muestra métricas y visualizaciones básicas que permiten comparar la regresión logística con gradient boosting, además de un panel para consultar probabilidades por fecha.

**Plan de mejoras futuras:**

1. Calibrar probabilidades mediante `CalibratedClassifierCV` con validación temporal (isotónica o Platt).
2. Ajustar el umbral de decisión optimizando *balanced accuracy* u otra métrica alineada al objetivo.
3. Implementar validación temporal robusta con `TimeSeriesSplit` y ventanas deslizantes.
4. Afinar hiperparámetros (p. ej. `C`, `learning_rate`, `max_depth`) con `GridSearchCV` respetando la temporalidad.
5. Incorporar variables de contexto (descanso, viajes, distancia, continuidad de plantilla, fuerza del rival).
6. Reducir ruido en `ROLL10` usando suavizados exponenciales (`ewm`) y `min_periods=5`.
7. Añadir métricas orientadas al negocio (ROC/PR detalladas, *lift*, ROI simulado, etc.).
8. Programar reentrenos periódicos (semanales) para seguir la deriva temporal y mantener la vigencia del modelo.
9. Mejorar la interpretabilidad con `permutation_importance` y visualizaciones SHAP que expliquen contribuciones puntuales.
